# Email Spam Classification: BoW vs. TF-IDF
**Category 03: Machine Learning | Project 03**

This walkthrough implements traditional machine learning text classification for email spam filtering.

### Key Highlights:
- **Text Sanitization**: Lowercasing, URL normalization, email stripping, punctuation filtering, and stopword removal.
- **Numerical Vectorization**: Contrasting **Bag of Words (CountVectorizer)** with **TF-IDF (TfidfVectorizer)**.
- **Classification Algorithms**: Benchmarking **Multinomial Naive Bayes**, **Logistic Regression**, and **Support Vector Machines** across both vectorizers.
- **Strict Leakage Prevention**: Fitting text vectorizers strictly on training data.
- **Multi-Metric Evaluation**: Precision, Recall, F1-Score, Accuracy, and Confusion Matrices.
- **Feature Attribution**: Identifying top discriminating tokens via model coefficients and log-likelihood ratios.
- **Inference & Threat Scoring**: Real-time email scoring with trigger token detection.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_email_data
from src.text_preprocessing import preprocess_corpus, split_email_dataset, clean_email_text
from src.feature_extraction import build_bow_vectorizer, build_tfidf_vectorizer, extract_top_indicative_keywords
from src.model_training import train_spam_classifiers, save_best_spam_pipeline
from src.model_evaluation import evaluate_spam_models, compare_vectorization_methods, plot_spam_evaluations
from src.predictor import SpamPredictor

print("Environment initialized successfully.")

## 1. Data Ingestion
Loading the email corpus containing legitimate workplace communications (Ham) and spam / phishing messages.

In [ ]:
raw_df = load_email_data()
print(f"Raw dataset dimensions: {raw_df.shape}")
raw_df.head(5)

## 2. Text Preprocessing & Cleaning
We remove duplicate emails, lowercase text, strip punctuation, filter stopwords, and normalize special tokens (URLs, numbers, currency).

In [ ]:
clean_df, report = preprocess_corpus(raw_df)
print("Preprocessing Report:", report)
clean_df[["message", "cleaned_message", "char_count", "word_count", "label_text"]].head(5)

## 3. Stratified Train-Test Splitting
We split the corpus into training (80%) and testing (20%) partitions with stratification on `label` to preserve spam proportions.

In [ ]:
X_train, X_test, y_train, y_test = split_email_dataset(clean_df, test_size=0.20, random_state=42)
print(f"Train samples: {len(X_train)} (Spam: {y_train.mean():.2%})")
print(f"Test samples:  {len(X_test)} (Spam: {y_test.mean():.2%})")

## 4. Model Training: BoW & TF-IDF x (Naive Bayes, Logistic Regression, SVM)
We train all six pipeline configurations on `X_train`:

In [ ]:
models = train_spam_classifiers(X_train, y_train, random_state=42)
for name in models:
    print(f"[+] Trained pipeline: {name}")

## 5. Performance Benchmarking & Vectorizer Comparison
Evaluating model predictions on the held-out test set.

In [ ]:
summary_df, predictions, probabilities = evaluate_spam_models(models, X_test, y_test)
display(summary_df)

vec_summary = compare_vectorization_methods(summary_df)
print("\n--- Vectorization Aggregates ---")
display(vec_summary)

## 6. Model Serialization & Interactive Threat Scoring
Serializing the top-performing pipeline and verifying production inference on sample emails.

In [ ]:
best_cfg = summary_df.iloc[0]["Configuration"]
best_pipe = models[best_cfg]
best_metrics = summary_df.iloc[0].to_dict()
saved_path = save_best_spam_pipeline(best_pipe, best_cfg, best_metrics)
print(f"Saved best model to: {saved_path}")

predictor = SpamPredictor(model_path=saved_path)

test_email = "URGENT: Claim your $5,000,000 cash inheritance lottery prize at http://claim-funds.xyz immediately!"
score = predictor.score_email(test_email, threshold=0.50)
for k, v in score.items():
    print(f"{k}: {v}")